In [1]:
import os
from pyspark.sql import SparkSession

# --- 1. Настройки окружения для macOS (Важно!) ---
# На macOS иногда возникают проблемы с форком процессов Java (Executor'ы не стартуют).
# Эта переменная часто решает проблему "Connection refused" или краши при запуске local-cluster
os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

# Очистка старых сессий и переменных (как у вас было)
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("✅ Существующая сессия остановлена.")
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Конфигурация Кластера ---
# Формат: local-cluster[число_воркеров, ядер_на_воркер, память_на_воркер_в_МБ]
# Мы просим 2 экзекутора, по 1 ядру, по 2 ГБ памяти каждый
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 6
MEMORY_PER_EXECUTOR_MB = 4096 

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"🚀 Запуск в режиме: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    # Память драйвера (остается у вас)
    .config("spark.driver.memory", "4g") 
    # Память экзекутора (должна соответствовать или быть меньше чем в master URL)
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "6")
    .config("spark.executor.instances", NUM_EXECUTORS)
    # Увеличиваем память под оверхед, чтобы избежать ошибок выделения памяти
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "4") # Для тестов меньше дефолтных 200
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")

# --- 3. Проверка конфигурации ---
print(f"✅ Сессия создана.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Проверка количества экзекуторов (может занять пару секунд на старт)
import time
time.sleep(3) 
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Активных экзекуторов (проверка через RDD): {num_executors}")

# --- 4. Тест на распределение (Пример) ---
# Чтобы убедиться, что задача ушла на экзекуторы, а не осталась на драйвере
def print_executor_info(iterator):
    import os
    # Получаем ID экзекутора из переменных окружения процесса
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Создаем датафрейм и применяем трансформацию
df = sp_s.range(0, 10, 1, 4) # 4 партиции
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Где выполнялись задачи:")
for line in result:
    print(line)

# Не забывайте останавливать сессию в конце скрипта, так как процессы тяжелые
# sp_s.stop() 

🚀 Запуск в режиме: local-cluster[2, 6, 4096]


26/04/28 16:33:09 WARN Utils: Your hostname, eric-Katana-17-B12UCR resolves to a loopback address: 127.0.1.1; using 10.246.79.83 instead (on interface wlo1)
26/04/28 16:33:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 16:33:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/28 16:33:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ Сессия создана.
Driver Memory Config: 4g
Executor Memory Config: 4g


📊 Активных экзекуторов (проверка через RDD): 2



🖥️ Где выполнялись задачи:
Executor ID: Driver/Local, PID: 294009
Executor ID: Driver/Local, PID: 294003
Executor ID: Driver/Local, PID: 294065
Executor ID: Driver/Local, PID: 294053


In [2]:
sp_s

In [3]:
sp_s.sparkContext.uiWebUrl

'http://10.246.79.83:4041'

In [4]:
import csv
import pandas as pd
import pyspark.pandas as ps
import warnings
from pyspark.pandas.utils import PandasAPIOnSparkAdviceWarning

warnings.filterwarnings("ignore", category=PandasAPIOnSparkAdviceWarning)
import random
from hypex import AATest
from hypex.analyzers.aa import AAScoreAnalyzer, OneAAStatAnalyzer
from hypex.comparators import StatsTTest, StatsChi2Test, GroupSizes, GroupTTest, GroupKSTest, GroupChi2Test, GroupDifference
from hypex.comparators.stats_hypothesis_testing import StatsTTest
from hypex.dataset import ExperimentData, AdditionalTreatmentRole
from hypex.dataset import ExperimentData, AdditionalTreatmentRole, TargetRole
from hypex.dataset import (
    Dataset, SmallDataset, InfoRole, TargetRole, 
    TreatmentRole, DatasetAdapter, DefaultRole, StratificationRole
)
from hypex.dataset.roles import TargetRole, FeatureRole, InfoRole, StatisticRole
from hypex.experiments.base import Experiment, OnRoleExperiment
from hypex.experiments.base_complex import ParamsExperiment
from hypex.reporters import DatasetReporter
from hypex.reporters.aa import OneAADictReporter
from hypex.splitters import AASplitter
from hypex.splitters.aa import AASplitter
from hypex.utils import BackendsEnum
from hypex.ui.aa import AAOutput
from hypex.ui.base import ExperimentShell

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [5]:
def generate_test_data(n_rows=100, filename=None, return_df=True):
    headers = ['user_id', 
               'signup_month', 
               'treat', 'pre_spends', 
               'post_spends', 'age', 'gender', 'industry']
    
    data = []
    current_id = 0
    
    while len(data) < n_rows:
        if current_id > 0 and current_id % 10 == 0:
            current_id += 1
            continue
            
        user_id = float(current_id)
        signup_month = float(random.randint(0, 11))
        treat = float(random.choice([0, 1]))
        
        pre_spends = random.uniform(450, 550)
        
        if treat == 0.0:
            post_spends = pre_spends * random.uniform(0.80, 0.90)
        else:
            post_spends = pre_spends * random.uniform(1.00, 1.10)
            
        age = float(random.randint(18, 70))
        gender = random.choice(['M', 'F'])
        industry = random.choice(['Logistics', 'E-commerce'])
        
        data.append([
            user_id, signup_month, treat, 
            round(pre_spends, 1), round(post_spends, 1), 
            age, gender, industry
        ])
        
        current_id += 1

    df = pd.DataFrame(data, columns=headers)
    
    if filename:
        df.to_csv(filename, index=False, encoding='utf-8')
        print(f"Сгенерировано {n_rows} строк в файле {filename}")
    
    if return_df:
        return df

In [6]:
def create_spark_ds(n_rows: int = 100):
    return Dataset(
        roles={
            "user_id": InfoRole(float),
            "treat": TreatmentRole(),
            "pre_spends": TargetRole(),
            "post_spends": DefaultRole(),
            "gender": StratificationRole(str),
            "industry" : DefaultRole()
        }, 
        data=generate_test_data(n_rows=n_rows), 
        session=sp_s,
        backend=BackendsEnum.spark
    )

In [7]:
from hypex.comparators.abstract import Comparator
from hypex.utils import SpaceEnum
from hypex.experiments import IfParamsExperiment
from hypex import AATest

In [8]:
ds = create_spark_ds(n_rows=1000)
ds.persist()
exp_data = ExperimentData(ds)
aa_test = AATest(n_iterations=3)
result = aa_test.execute(exp_data)
exp_data.analysis_tables

ParamsExperiment <class 'hypex.dataset.backends.spark_backend.SparkDataset'>
ParamsExperiment


  0%|          | 0/3 [00:00<?, ?it/s]

AASplitter <class 'hypex.dataset.backends.spark_backend.SparkDataset'>
AASplitter
executor.key = AASplitter┴rs 0┴; dt = 6.7699c
Experiment <class 'hypex.dataset.backends.spark_backend.SparkDataset'>
Experiment
GroupSizes <class 'hypex.dataset.backends.spark_backend.SparkDataset'>
GroupSizes
executor.key = GroupSizes┴┴AASplitter|rs 0|; dt = 4.5235c
OnRoleExperiment <class 'hypex.dataset.backends.spark_backend.SparkDataset'>
OnRoleExperiment
TTest <class 'hypex.dataset.backends.spark_backend.SparkDataset'>
StatsTTest


  0%|          | 0/3 [00:13<?, ?it/s]


NotSuitableFieldError: Grouping field Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4, ..., 995, 996, 997, 998, 999]

1000 rows × 0 columns is not suitable for comparison

In [ ]:
Dataset.create_empty(
    roles={"result": StatisticRole()}, backend=BackendsEnum.pandas
)

In [ ]:
ds = create_spark_ds(n_rows=1000)
ds.persist()
exp_data = ExperimentData(ds)

AA_METRICS = Experiment(
    executors=[
        OnRoleExperiment(
            executors=[
                GroupDifference(compare_by="groups", grouping_role=AdditionalTreatmentRole()),
                StatsTTest(grouping_role=AdditionalTreatmentRole(),
                           target_roles=TargetRole(),
                           reliability=0.05)
            ],
            role=TargetRole(),
        ),
        OneAAStatAnalyzer(),
    ]
)

ONE_AA_TEST = Experiment(executors=[AASplitter(), AA_METRICS])

AA_TEST = Experiment(
    [
        ParamsExperiment(
            executors=([ONE_AA_TEST]),
            params={
                AASplitter: {"random_state": range(3), "control_size": [0.5]},
                Comparator: {
                    "grouping_role": [AdditionalTreatmentRole()],
                    "space": [SpaceEnum.additional],
                },
            },
            reporter=DatasetReporter(OneAADictReporter(front=False)),
        ),
        AAScoreAnalyzer(),
    ],
    key="AATest",
)

AA_TEST.execute(exp_data)
exp_data.analysis_tables

In [ ]:
# type(exp_data.analysis_tables["ParamsExperiment┴┴AATest"])

In [ ]:
from hypex.dataset import ReportRole
from hypex.utils import ID_SPLIT_SYMBOL
pass_suffix = f"{ID_SPLIT_SYMBOL}pass{ID_SPLIT_SYMBOL}all"
pval_suffix = f"{ID_SPLIT_SYMBOL}p-value{ID_SPLIT_SYMBOL}all"

In [ ]:
score_table = SmallDataset.from_dict(data={
    "splitter_id": ["AASplitter┴rs 0┴", "AASplitter┴rs 1┴", "AASplitter┴rs 2┴"],
    f"mean{ID_SPLIT_SYMBOL}StatsTTest{ID_SPLIT_SYMBOL}p-value": [0.400203, 0.201783, 0.913649],
    f"mean{ID_SPLIT_SYMBOL}StatsTTest{ID_SPLIT_SYMBOL}pass": [0.0, 0.0, 0.0],
    f"mean{ID_SPLIT_SYMBOL}test{ID_SPLIT_SYMBOL}score": [0.133401, 0.067261, 0.304550]},
    roles={'splitter_id': ReportRole(), 
           f"mean{ID_SPLIT_SYMBOL}StatsTTest{ID_SPLIT_SYMBOL}p-value": ReportRole(), 
           f"mean{ID_SPLIT_SYMBOL}StatsTTest{ID_SPLIT_SYMBOL}pass": ReportRole(), 
           f"mean{ID_SPLIT_SYMBOL}test{ID_SPLIT_SYMBOL}score": ReportRole()})

In [ ]:
(score_table[["mean┴StatsTTest┴p-value",	"mean┴StatsTTest┴pass",	"mean┴test┴score"]] + 
 score_table[["mean┴StatsTTest┴p-value",	"mean┴StatsTTest┴pass",	"mean┴test┴score"]])

In [ ]:
score_table.columns

In [ ]:
score_table = SmallDataset.from_dict(data={
    "splitter_id": ["AASplitter┴rs 0┴", "AASplitter┴rs 1┴", "AASplitter┴rs 2┴"],
    f"mean{ID_SPLIT_SYMBOL}StatsTTest{ID_SPLIT_SYMBOL}p-value": [0.400203, 0.201783, 0.913649],
    f"mean{ID_SPLIT_SYMBOL}StatsTTest{ID_SPLIT_SYMBOL}pass": [0.0, 0.0, 0.0],
    f"mean{ID_SPLIT_SYMBOL}test{ID_SPLIT_SYMBOL}score": [0.133401, 0.067261, 0.304550]},
    roles={'splitter_id': ReportRole(), 
           f"mean{ID_SPLIT_SYMBOL}StatsTTest{ID_SPLIT_SYMBOL}p-value": ReportRole(), 
           f"mean{ID_SPLIT_SYMBOL}StatsTTest{ID_SPLIT_SYMBOL}pass": ReportRole(), 
           f"mean{ID_SPLIT_SYMBOL}test{ID_SPLIT_SYMBOL}score": ReportRole()})

print(f"select type = {type(score_table.select('mean┴StatsTTest┴p-value'))}")
score_table.select("mean┴StatsTTest┴p-value") # работает
score_table.select("mean┴StatsTTest┴p-value") * 2 # ошибка

In [ ]:
score_table["mean┴StatsTTest┴p-value"]

In [ ]:
score_table

In [ ]:
pvalue_weight = 2 / 3
test_score_weight = 1 / 3

In [ ]:
[('mean┴StatsTTest┴pass┴all', 0.95)]

In [ ]:
weighted_pvalues = None
for key, weight in [('mean┴StatsTTest┴pass┴all', 0.95)]:
    if weight > 0:
        pval_col = key.replace(pass_suffix, pval_suffix)
        print(pval_col)
        print(score_table.columns)
        if pval_col in score_table.columns:
            print(2)
            col_data = score_table.select(pval_col)
            print(f"pval_col = {pval_col}")
            print(f"col_data columns = {col_data.columns}")
            if weighted_pvalues is None:
                weighted_pvalues = col_data * weight
            else:
                weighted_pvalues = weighted_pvalues + col_data * weight

In [ ]:
score_table.columns

In [ ]:
weighted_pvalues

In [ ]:
ds = create_spark_ds(n_rows=1000)
ds.persist()
exp_data = ExperimentData(ds)

splitter = AASplitter(save_groups=True)
splitter.execute(exp_data)

group_sizes = GroupSizes(grouping_role=AdditionalTreatmentRole())
group_sizes.execute(exp_data)



ds.unpersist()

In [ ]:
exp_data.analysis_tables["GroupSizes┴┴AASplitter||"]

In [ ]:
for key, table in exp_data.analysis_tables.items():
    print(f"{key}: {table.to_dict()}")

In [ ]:
exp_data.groups

In [ ]:
exp_data.ds[exp_data.additional_fields["AASplitter┴┴"] == "test_0"]

In [ ]:
exp_data.additional_fields.filter()

In [ ]:
list(exp_data.additional_fields.unique()["AASplitter┴┴"].to_dict().values())

In [ ]:
ds.roles

In [ ]:
# ttest = StatsTTest(grouping_role=AdditionalTreatmentRole(),
#                    target_roles=TargetRole(),
#                    reliability=0.05)
# ttest.execute(exp_data)

# one_aa_stat_anal = OneAAStatAnalyzer()
# one_aa_stat_anal.execute(exp_data)

In [ ]:
exp_data.groups

In [ ]:
type(exp_data.analysis_tables["StatsTTest┴┴pre_spends┆stats"])

In [ ]:
exp_data.analysis_tables["OneAAStatAnalyzer┴┴"]

In [ ]:
exp_data.groups

In [ ]:
ds.roles

In [ ]:
exp_data.analysis_tables["OneAAStatAnalyzer┴┴"]

In [ ]:
from hypex.utils import ID_SPLIT_SYMBOL, BackendsEnum, ExperimentDataEnum

In [ ]:
analysis_tests: list[type] = [GroupTTest, GroupKSTest, GroupChi2Test,
                              StatsTTest, StatsChi2Test]
executor_ids = exp_data.get_ids(
    analysis_tests, searched_space=ExperimentDataEnum.analysis_tables
)
executor_ids

In [ ]:
analysis_data: dict[str, float] = {}
for class_, spaces in executor_ids.items():
    analysis_ids = spaces.get("analysis_tables", [])
    if len(analysis_ids) > 0:
        if len(analysis_ids) > 1:
            t_data = exp_data.analysis_tables[analysis_ids[0]].append(
                [exp_data.analysis_tables[k] for k in analysis_ids[1:]]
            )
        else:
            t_data = exp_data.analysis_tables[analysis_ids[0]]
        # t_data.data.index = analysis_ids
        for field in ["p-value", "pass"]:
            analysis_data[f"mean {class_} {field}"] = t_data[field].mean()
analysis_data

In [ ]:
import numpy as np

analysis_data["mean test score"] = 0
sum_weight = 0
analysis_data = {
    key: (0 if np.isnan(value) else value)
    for key, value in analysis_data.items()
}

In [ ]:
analysis_data

In [ ]:
exp_data.analysis_tables

In [ ]:
def AATest_without_analyse(num_splits: int = 1) -> ParamsExperiment:
    base_experiment = Experiment(
        executors=[
            AASplitter(
                control_size=0.5, 
                constant_key=True,
                save_groups=True
            ),
            StatsTTest(
                grouping_role=AdditionalTreatmentRole(),
                target_roles=TargetRole(),
                reliability=0.05
            ),
            # StatsChi2Test(
            #     grouping_role=AdditionalTreatmentRole(),
            #     target_roles=StratificationRole(),
            #     reliability=0.05
            # ),
            OneAAStatAnalyzer()
        ]
    )

    return ParamsExperiment(
        executors=[base_experiment],
        params={
            AASplitter: {
                "random_state": range(num_splits),
                "control_size": [0.5]
            }
        },
        reporter=DatasetReporter(OneAADictReporter(front=False))
    )

In [ ]:
ds = create_spark_ds(n_rows=1_000)
ds.persist()
exp_data = ExperimentData(ds)
AATest_without_analyse().execute(exp_data)
ds.unpersist()

In [ ]:
exp_data.analysis_tables

In [ ]:
def AATest_without_analyse(num_splits: int = 10) -> Experiment:
    base_executors = [
        AASplitter(control_size=0.5, constant_key=True, save_groups=True),
        GroupSizes(grouping_role=AdditionalTreatmentRole()),
        StatsTTest(grouping_role=AdditionalTreatmentRole(), target_roles=TargetRole(), reliability=0.05),
        StatsChi2Test(grouping_role=AdditionalTreatmentRole(), target_roles=StratificationRole(), reliability=0.05),
        OneAAStatAnalyzer()
    ]

    return Experiment(
        executors=[
            ParamsExperiment(
                executors=[Experiment(executors=base_executors)],
                params={AASplitter: {"random_state": range(num_splits), "control_size": [0.5]}},
                reporter=DatasetReporter(OneAADictReporter(front=False))
            ),
            AAScoreAnalyzer()
        ],
        key="AATest"
    )

In [ ]:
exp_data.

In [ ]:
exp_data.additional_fields

In [ ]:
exp_data = ExperimentData(create_spark_ds(n_rows=10000))
# Создаём и запускаем сплиттер
splitter = AASplitter(
    control_size=0.5,
    random_state=21,
    constant_key=True,
    save_groups=False  # Сохраняем группы в data.groups для удобного доступа
)
result = splitter.execute(exp_data)

In [ ]:
print("Разбиение выполнено!")
print(f"ID сплиттера: {splitter.id}")

test = StatsTTest(
    grouping_role=AdditionalTreatmentRole(),
    target_roles=TargetRole(),
    reliability=0.05
)
result_test = test.execute(result)

In [ ]:
# Просмотр результатов т-теста
print("Результаты т-теста:")
for table_id, table in result_test.analysis_tables.items():
    print(f"\nТаблица {table_id}:")
    print(table)

In [ ]:
test = StatsChi2Test(
    grouping_role=AdditionalTreatmentRole(),
    target_roles=StratificationRole(),
    reliability=0.05
)
result_test = test.execute(result)

In [ ]:
# Просмотр результатов т-теста
print("Результаты т-теста:")
for table_id, table in result_test.analysis_tables.items():
    print(f"\nТаблица {table_id}:")
    print(table)

In [ ]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import psutil
import os
import gc
from datetime import datetime

In [ ]:
# ================= КОНФИГУРАЦИЯ =================
class BenchmarkConfig:
    # N_ROWS_VALUES = [1000, 5000, 10000, 50000, 100000, 500000, 1000000, 5000000, 10000000][::-1]
    N_ROWS_VALUES = [1000, 5000, 10000, 50000, 100000]

    N_REPEATS = 1
    MEMORY_SAMPLE_INTERVAL = 0.5  # сек
    SAVE_PATH = 'spark_benchmark_results'
    
    @classmethod
    def ensure_dir(cls):
        os.makedirs(cls.SAVE_PATH, exist_ok=True)

In [ ]:
# ================= МОНТОРИНГ ПАМЯТИ =================
class MemoryMonitor:
    """Сборщик метрик памяти (JVM + System + Spark)"""
    
    def __init__(self, spark_session):
        self.spark = spark_session
        self.jvm = spark_session._jvm
        self.process = psutil.Process(os.getpid())
        self.samples = []
        self.is_monitoring = False
        
    def start(self):
        self.samples = []
        self.is_monitoring = True
        
    def stop(self):
        self.is_monitoring = False
        
    def sample(self):
        """Снимает текущий срез метрик памяти"""
        if not self.is_monitoring:
            return
            
        try:
            runtime = self.jvm.Runtime.getRuntime()
            
            sample = {
                'timestamp': time.time(),
                # JVM Heap
                'jvm_heap_used_mb': (runtime.totalMemory() - runtime.freeMemory()) / 1024 / 1024,
                'jvm_heap_max_mb': runtime.maxMemory() / 1024 / 1024,
                'jvm_heap_committed_mb': runtime.totalMemory() / 1024 / 1024,
                
                # System RSS
                'process_rss_mb': self.process.memory_info().rss / 1024 / 1024,
                'process_vms_mb': self.process.memory_info().vms / 1024 / 1024,
                
                # CPU
                'cpu_percent': self.process.cpu_percent(interval=0.1),
            }
            
            # Spark-метрики (если доступны)
            try:
                sc = self.spark.sparkContext
                if hasattr(sc, '_jsc'):
                    jsc = sc._jsc
                    # Статистика пула (упрощенно)
                    sample['spark_executors'] = len(sc._jsc.sc().executorIds())
            except:
                sample['spark_executors'] = 0
                
            self.samples.append(sample)
            
        except Exception as e:
            print(f"⚠️ Ошибка сбора метрик: {e}")
    
    def get_statistics(self):
        """Агрегирует статистику по собранным сэмплам"""
        if not self.samples:
            return {}
            
        df = pd.DataFrame(self.samples)
        
        stats = {
            'jvm_heap_used_avg_mb': df['jvm_heap_used_mb'].mean(),
            'jvm_heap_used_max_mb': df['jvm_heap_used_mb'].max(),
            'jvm_heap_used_min_mb': df['jvm_heap_used_mb'].min(),
            'process_rss_avg_mb': df['process_rss_mb'].mean(),
            'process_rss_max_mb': df['process_rss_mb'].max(),
            'cpu_avg_percent': df['cpu_percent'].mean(),
            'cpu_max_percent': df['cpu_percent'].max(),
            'samples_count': len(df),
        }
        return stats
    
    def plot_memory_timeline(self, ax):
        """Рисует таймлайн потребления памяти"""
        if not self.samples:
            return
            
        df = pd.DataFrame(self.samples)
        df['relative_time'] = df['timestamp'] - df['timestamp'].min()
        
        ax.plot(df['relative_time'], df['jvm_heap_used_mb'], label='JVM Heap Used', color='red', linewidth=2)
        ax.plot(df['relative_time'], df['process_rss_mb'], label='Process RSS', color='blue', linewidth=2)
        ax.axhline(y=df['jvm_heap_max_mb'].iloc[0], color='red', linestyle='--', alpha=0.5, label='JVM Max')
        
        ax.set_xlabel('Время (сек)')
        ax.set_ylabel('Память (MB)')
        ax.set_title('Динамика потребления памяти')
        ax.legend(loc='upper left')
        ax.grid(True, alpha=0.3)

In [ ]:
# ================= ФОНОВЫЙ СБОРЩИК =================
class BackgroundMonitor:
    """Запускает сбор метрик в фоновом потоке"""
    
    def __init__(self, memory_monitor, interval=0.5):
        self.monitor = memory_monitor
        self.interval = interval
        self.running = False
        
    def start(self):
        self.running = True
        self.monitor.start()
        
    def stop(self):
        self.running = False
        self.monitor.stop()
        
    def run_and_collect(self, func, *args, **kwargs):
        """Запускает функцию и параллельно собирает метрики"""
        import threading
        
        def collect_loop():
            while self.running:
                self.monitor.sample()
                time.sleep(self.interval)
        
        # Запуск потока сбора
        collector = threading.Thread(target=collect_loop, daemon=True)
        self.start()
        collector.start()
        
        # Выполнение целевой функции
        result = func(*args, **kwargs)
        
        # Остановка
        self.stop()
        collector.join(timeout=1)
        
        return result, self.monitor.get_statistics()

In [ ]:

# ================= БЕНЧМАРК =================
def run_single_test(spark_ds, spark_session, monitor):
    """Выполняет один тестовый прогон"""
    exp_data = ExperimentData(spark_ds)
    AATest_without_analyse().execute(exp_data)

In [ ]:
def run_benchmark_with_memory(spark_session):
    """Полный бенчмарк с памятью"""
    BenchmarkConfig.ensure_dir()
    monitor = MemoryMonitor(spark_session)
    bg_monitor = BackgroundMonitor(monitor, interval=BenchmarkConfig.MEMORY_SAMPLE_INTERVAL)
    
    results = []
    print(f"🚀 Старт бенчмарка: {datetime.now().strftime('%H:%M:%S')}")
    print(f"💾 JVM Max Heap: {spark_session._jvm.Runtime.getRuntime().maxMemory() / 1024 / 1024 / 1024:.2f} GB")
    print("-" * 70)
    
    for n_rows in BenchmarkConfig.N_ROWS_VALUES:
        spark_ds = create_spark_ds(n_rows=n_rows)
        print(f"\n📊 n_rows = {n_rows:,}")
        
        gc.collect()
        time.sleep(0.5)
        
        times = []
        memory_stats_list = []
        
        for i in range(BenchmarkConfig.N_REPEATS):
            try:
                start = time.perf_counter()
                
                _, mem_stats = bg_monitor.run_and_collect(
                    run_single_test, spark_ds, spark_session, monitor
                )
                
                end = time.perf_counter()
                duration = end - start
                times.append(duration)
                memory_stats_list.append(mem_stats)
                
                print(f"   Повтор {i+1}: {duration:.3f} сек | "
                      f"RSS Max: {mem_stats.get('process_rss_max_mb', 0):.0f} MB")
                
            except Exception as e:
                print(f"   ❌ Ошибка: {e}")
                times.append(None)
                memory_stats_list.append({})
        
        valid_times = [t for t in times if t is not None]
        if valid_times:
            avg_mem_stats = {
                k: np.mean([s.get(k, 0) for s in memory_stats_list]) 
                for k in memory_stats_list[0].keys() if k != 'samples_count'
            }
            max_mem_stats = {
                k: np.max([s.get(k, 0) for s in memory_stats_list]) 
                for k in memory_stats_list[0].keys() if k != 'samples_count'
            }
            
            # === ИСПРАВЛЕНИЕ: Считаем mb_per_million сразу здесь ===
            rss_max = max_mem_stats.get('process_rss_max_mb', 0)
            mb_per_million = (rss_max / n_rows * 1_000_000) if n_rows > 0 else 0
            
            results.append({
                'n_rows': n_rows,
                'time_mean_sec': np.mean(valid_times),
                'time_std_sec': np.std(valid_times),
                'jvm_heap_avg_mb': avg_mem_stats.get('jvm_heap_used_avg_mb', 0),
                'jvm_heap_max_mb': max_mem_stats.get('jvm_heap_used_max_mb', 0),
                'process_rss_avg_mb': avg_mem_stats.get('process_rss_avg_mb', 0),
                'process_rss_max_mb': rss_max,
                'cpu_avg_percent': avg_mem_stats.get('cpu_avg_percent', 0),
                'rows_per_sec': n_rows / np.mean(valid_times),
                'mb_per_million': mb_per_million,  # Добавлено в результат
            })
        else:
            results.append({'n_rows': n_rows, 'time_mean_sec': None})
    
    print(f"\n🏁 Завершено: {datetime.now().strftime('%H:%M:%S')}")
    return pd.DataFrame(results)

In [ ]:
# ================= ВИЗУАЛИЗАЦИЯ =================
def plot_comprehensive_results(df):
    """Строит расширенные графики"""
    df_clean = df.dropna(subset=['time_mean_sec']).copy()
    if len(df_clean) == 0:
        print("⚠️ Нет данных для графиков")
        return

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('📈 Комплексный анализ производительности и памяти', fontsize=14, fontweight='bold')
    
    # 1. Время выполнения
    ax1 = axes[0, 0]
    ax1.errorbar(df_clean['n_rows'], df_clean['time_mean_sec'], 
                 yerr=df_clean['time_std_sec'], marker='o', capsize=5)
    ax1.set_xscale('log')
    ax1.set_xlabel('n_rows')
    ax1.set_ylabel('Время (сек)')
    ax1.set_title('Время выполнения')
    ax1.grid(True, alpha=0.3)
    
    # 2. JVM Память
    ax2 = axes[0, 1]
    ax2.plot(df_clean['n_rows'], df_clean['jvm_heap_avg_mb'], marker='s', label='Avg', color='red')
    ax2.plot(df_clean['n_rows'], df_clean['jvm_heap_max_mb'], marker='^', label='Max', color='darkred')
    ax2.set_xscale('log')
    ax2.set_xlabel('n_rows')
    ax2.set_ylabel('Память (MB)')
    ax2.set_title('Потребление JVM Heap')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Системная память (RSS)
    ax3 = axes[0, 2]
    ax3.plot(df_clean['n_rows'], df_clean['process_rss_avg_mb'], marker='d', label='RSS Avg', color='blue')
    ax3.plot(df_clean['n_rows'], df_clean['process_rss_max_mb'], marker='x', label='RSS Max', color='navy')
    ax3.set_xscale('log')
    ax3.set_xlabel('n_rows')
    ax3.set_ylabel('Память (MB)')
    ax3.set_title('Системная память процесса')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Производительность
    ax4 = axes[1, 0]
    ax4.plot(df_clean['n_rows'], df_clean['rows_per_sec'], marker='o', color='green')
    ax4.set_xscale('log')
    ax4.set_xlabel('n_rows')
    ax4.set_ylabel('строк/сек')
    ax4.set_title('Пропускная способность')
    ax4.grid(True, alpha=0.3)
    
    # 5. Память на 1 млн строк (эффективность)
    ax5 = axes[1, 1]
    # Проверка на наличие колонки на случай загрузки старого CSV
    if 'mb_per_million' in df_clean.columns:
        data_to_plot = df_clean['mb_per_million']
    else:
        # Фолбэк если колонки нет (пересчет)
        data_to_plot = df_clean['process_rss_max_mb'] / df_clean['n_rows'] * 1_000_000
        
    ax5.bar(range(len(df_clean)), data_to_plot, color='orange')
    ax5.set_xticks(range(len(df_clean)))
    ax5.set_xticklabels([f"{n/1e6:.1f}M" for n in df_clean['n_rows']], rotation=45)
    ax5.set_ylabel('MB на 1 млн строк')
    ax5.set_title('Эффективность использования памяти')
    ax5.grid(True, alpha=0.3, axis='y')
    
    # 6. CPU загрузка
    ax6 = axes[1, 2]
    ax6.plot(df_clean['n_rows'], df_clean['cpu_avg_percent'], marker='*', color='purple')
    ax6.set_xscale('log')
    ax6.set_xlabel('n_rows')
    ax6.set_ylabel('CPU %')
    ax6.set_title('Загрузка CPU')
    ax6.set_ylim(0, 100)
    ax6.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{BenchmarkConfig.SAVE_PATH}/full_analysis.png', dpi=300, bbox_inches='tight')
    print(f"📁 Графики сохранены в {BenchmarkConfig.SAVE_PATH}/full_analysis.png")
    plt.show()

In [ ]:
def print_memory_analysis(df):
    """Выводит текстовый анализ памяти"""
    df_clean = df.dropna(subset=['time_mean_sec']).copy()
    if len(df_clean) < 2:
        print("⚠️ Недостаточно данных для анализа памяти")
        return

    print("\n" + "="*70)
    print("🔍 АНАЛИЗ ПОТРЕБЛЕНИЯ ПАМЯТИ")
    print("="*70)
    
    first, last = df_clean.iloc[0], df_clean.iloc[-1]
    
    # Рост памяти
    mem_growth = last['process_rss_max_mb'] / first['process_rss_max_mb'] if first['process_rss_max_mb'] > 0 else 0
    data_growth = last['n_rows'] / first['n_rows']
    
    print(f"\n📏 Масштабирование памяти:")
    print(f"   Данные: ×{data_growth:,.1f}")
    print(f"   Память (RSS): ×{mem_growth:.2f}")
    
    if mem_growth > data_growth * 1.5:
        print("   ⚠️ Память растет быстрее данных! Возможна утечка или неоптимальные структуры.")
    else:
        print("   ✓ Память масштабируется линейно или лучше.")
    
    # Пики потребления
    print(f"\n📊 Пиковые значения:")
    print(f"   Max JVM Heap: {df_clean['jvm_heap_max_mb'].max():.0f} MB "
          f"({df_clean['jvm_heap_max_mb'].max()/1024:.2f} GB)")
    print(f"   Max Process RSS: {df_clean['process_rss_max_mb'].max():.0f} MB "
          f"({df_clean['process_rss_max_mb'].max()/1024:.2f} GB)")
    
    # Эффективность (с проверкой существования колонки)
    if 'mb_per_million' in df_clean.columns:
        avg_mb_per_million = df_clean['mb_per_million'].mean()
    else:
        # Пересчет если колонки нет
        avg_mb_per_million = (df_clean['process_rss_max_mb'] / df_clean['n_rows'] * 1_000_000).mean()
        
    print(f"\n⚡ Эффективность:")
    print(f"   Среднее потребление: {avg_mb_per_million:.2f} MB на 1 млн строк")
    
    # Рекомендации
    print(f"\n💡 Рекомендации по памяти:")
    max_rss_gb = df_clean['process_rss_max_mb'].max() / 1024
    if max_rss_gb > 8:
        print(f"   • Высокое потребление ({max_rss_gb:.1f} GB). Увеличьте `spark.driver.memory`.")
    if 'cpu_avg_percent' in df_clean.columns and df_clean['cpu_avg_percent'].mean() < 50:
        print(f"   • Низкая загрузка CPU. Можно увеличить параллелизм.")

In [ ]:
# 2. Запуск бенчмарка
results_df = run_benchmark_with_memory(sp_s)

# 3. Сохранение данных
results_df.to_csv(f'{BenchmarkConfig.SAVE_PATH}/benchmark_data.csv', index=False)

In [ ]:
# 4. Визуализация и анализ
plot_comprehensive_results(results_df.sort_values(by="n_rows"))
print_memory_analysis(results_df.sort_values(by="n_rows"))

In [ ]:
f'{BenchmarkConfig.SAVE_PATH}/benchmark_data.csv'